# Mapping migration


## Set up

To get started on this notebook, you’ll need to restore any variables from previous notebooks to your workspace. To save time and memory, make sure to specify which variables you want to load.

In [24]:
%store -r

You will also need to import any libraries you are using in this notebook. In this notebook, you will need some additional packages to make interactive maps with vector data and access pre-defined month names.

In [25]:
### import packages

### libraries for dynamic mapping

### manage coordinate ref systems
import cartopy.crs as ccrs

### customize dynamic plot
import panel as pn

### make dynamic plot with geodataframe
import hvplot.pandas
import holoviews as hv

### Run this fix if the plot is not showing, once per kernal session
hvplot.extension("bokeh")

### get month names
import calendar

### folder/file management
import os
import pathlib



## Step 5: Make an interactive map of GBIF observations

### Step 5A: Create a simplified `GeoDataFrame` for plotting

Plotting larger files can be time consuming. The code below will streamline plotting with `hvplot` by simplifying the geometry, projecting it to a Mercator projection that is compatible with `geoviews`, and cropping off areas in the Arctic:

1. Simplify the ecoregions with `.simplify(.05)`, and save it back to the `geometry` column.
2. Change the Coordinate Reference System (CRS) to Mercator with `.to_crs(ccrs.Mercator())`.
3. Use the plotting code that is already in the cell to check that the plotting runs quickly (less than a minute) and looks the way you want, making sure to change `gdf` to your `GeoDataFrame` name.

In [26]:
### simplify the geometry to speed up processing
ecoreg_gdf.geometry = ecoreg_gdf.geometry.simplify(
    0.05, preserve_topology=False)

### change the CRS to Mercator for mapping
ecoreg_gdf = ecoreg_gdf.to_crs(ccrs.Mercator())

### check that the plot runs in a reasonable amount of time
ecoreg_gdf.hvplot(geo = True, crs = ccrs.Mercator())

:Polygons   [Longitude,Latitude]

### Step 5B: Map migration over time

1. If applicable, replace any variable names with the names you defined previously.
2. Replace `column_name_used_for_ecoregion_color` and `column_name_used_for_slider` with the column names you wish to use.
3. Customize your plot with your choice of title, tile source, color map, and size.

Your plot will probably still change months very slowly in your Jupyter notebook, because it calculates each month’s plot as needed. Open up the saved HTML file to see faster performance!

In [ ]:
### prep the data, then combine the occurrence data with the ecoregions

### rename columns in ecoregID to match occurrence_df
ecoreg_gdf = ecoreg_gdf.rename(columns = {'OBJECTID': 'ecoregion'})

### make ecoreg ID the index for merge with occurrence_df
### Run this once but comment out if you rerun
ecoreg_gdf = ecoreg_gdf.set_index('ecoregion')

### join the occurrences with the plotting GeoDataFrame
occurrence_gdf = ecoreg_gdf.join(occurrence_df)

occurrence_gdf


ECO_NAME  BIOME_NUM  \
ecoregion month                                                          
13.0      5      Alberta-British Columbia foothills forests        5.0   
          6      Alberta-British Columbia foothills forests        5.0   
          7      Alberta-British Columbia foothills forests        5.0   
17.0      4                     Allegheny Highlands forests        4.0   
          5                     Allegheny Highlands forests        4.0   
...                                                     ...        ...   
839.0     7                Northern Rockies conifer forests        5.0   
          8                Northern Rockies conifer forests        5.0   
          9                Northern Rockies conifer forests        5.0   
845.0     9              North Atlantic moist mixed forests        4.0   
          10             North Atlantic moist mixed forests        4.0   

                                          BIOME_NAME       REALM ECO_BIOME_  \
ecoregion month                                                               
13.0      5                Temperate Conifer Forests    Nearctic       NE05   
          6                Temperate Conifer Forests    Nearctic       NE05   
          7                Temperate Conifer Forests    Nearctic       NE05   
17.0      4      Temperate Broadleaf & Mixed Forests    Nearctic       NE04   
          5      Temperate Broadleaf & Mixed Forests    Nearctic       NE04   
...                                              ...         ...        ...   
839.0     7                Temperate Conifer Forests    Nearctic       NE05   
          8                Temperate Conifer Forests    Nearctic       NE05   
          9                Temperate Conifer Forests    Nearctic       NE05   
845.0     9      Temperate Broadleaf & Mixed Forests  Palearctic       PA04   
          10     Temperate Broadleaf & Mixed Forests  Palearctic       PA04   

                 NNH  ECO_ID  SHAPE_LENG  SHAPE_AREA  \
ecoregion month                                        
13.0      5        2     345   62.333821   17.133639   
          6        2     345   62.333821   17.133639   
          7        2     345   62.333821   17.133639   
17.0      4        2     328   24.038587    7.958751   
          5        2     328   24.038587    7.958751   
...              ...     ...         ...         ...   
839.0     7        2     361   56.924527   35.905513   
          8        2     361   56.924527   35.905513   
          9        2     361   56.924527   35.905513   
845.0     9        3     672   89.144126    5.586107   
          10       3     672   89.144126    5.586107   

                                          NNH_NAME    COLOR COLOR_BIO  \
ecoregion month                                                         
13.0      5      Nature Could Reach Half Protected  #5DAD4C   #458970   
          6      Nature Could Reach Half Protected  #5DAD4C   #458970   
          7      Nature Could Reach Half Protected  #5DAD4C   #458970   
17.0      4      Nature Could Reach Half Protected  #C8EBB1   #00734C   
          5      Nature Could Reach Half Protected  #C8EBB1   #00734C   
...                                            ...      ...       ...   
839.0     7      Nature Could Reach Half Protected  #ACC13E   #458970   
          8      Nature Could Reach Half Protected  #ACC13E   #458970   
          9      Nature Could Reach Half Protected  #ACC13E   #458970   
845.0     9                   Nature Could Recover  #378F52   #00734C   
          10                  Nature Could Recover  #378F52   #00734C   

                COLOR_NNH    LICENSE  \
ecoregion month                        
13.0      5       #7BC141  CC-BY 4.0   
          6       #7BC141  CC-BY 4.0   
          7       #7BC141  CC-BY 4.0   
17.0      4       #7BC141  CC-BY 4.0   
          5       #7BC141  CC-BY 4.0   
...                   ...        ...   
839.0     7       #7BC141  CC-BY 4.0   
          8       #7B

In [29]:
### activate panel and hvplot extensions you'll need for the interactive plot
pn.extension()
hv.extension("bokeh")

In [31]:
### plot the data

### get the plot bounds so they don't change with the slider
xmin, ymin, xmax, ymax = occurrence_gdf.total_bounds

### define the slider widget
slider = pn.widgets.DiscreteSlider(
    name = 'month', 
    options = {calendar.month_name[i]: i for i in range(1, 13)}
)

### plot occurrence by ecoregion and month
migration_plot = (
    occurrence_gdf
    .hvplot(
        ### Color by number veeries
        c = 'norm_occurrences',
        groupby = 'month',

        ### use background tiles
        geo = True, crs = ccrs.Mercator(), tiles = 'OSM',

        ### add a title
        title = "Yearly Veery Migration",

        ### set plot boundaries
        xlim = (xmin, xmax), ylim = (ymin, ymax),

        ### set dimensions of map
        frame_height = 600,
        frame_width = 900,

        ### put the widget at the bottom of the map
        widget_location = 'bottom',
        colorbar = False,
        widgets = {'column_name': slider}
    )
)

### wrap  plot in the Panel layout
interactive_migration_plot = pn.panel(migration_plot)

### check it out
interactive_migration_plot

BokehModel(combine_events=True, render_bundle={'docs_json': {'f51e0702-2c42-4a52-8c06-4742bb3d731f': {'version…

Due to some annoying issues with permissions for the basemap, we can't export the map you just made as an .html file with the basemap. So we'll make another version, without a basemap, to export. We'll revisit this for your portfolio assignment to make a nice map with a basemap.

In [ ]:
### make a path for your map to live
map_path = os.path.join(data_dir, "04_veery_plot.html")

map_path

In [ ]:
### make a version for export, without the basemap
export_migration_plot = (
    occurrence_gdf
    .hvplot(
        ### Color by number veeries
        c = 'norm_occurrences',
        groupby = 'month',

        ### add a title
        title = "Yearly Veery Migration",

        ### set plot boundaries
        xlim = (xmin, xmax), ylim = (ymin, ymax),

        ### set dimensions of map
        frame_height = 600,
        frame_width = 900,

        ### put the widget at the bottom of the map
        widget_location = 'bottom',
        colorbar = False,
        widgets = {'column_name': slider}
    )
)

In [ ]:
### export the map as an .html file
export_migration_plot.save(
    map_path,
    embed = True,
    resources = "inline",
)

print(f"Map saved to {map_path}")

### Reflect and Respond #2: Interpret your map
What seasonal trends do you see in the GBIF observations? How does your map align with the broader literature on Veery migration?